# ENRICH-Aerial — GCP-controlled bundle adjustment


In [1]:
from pathlib import Path

import enlighten
import numpy as np
import pycolmap
from pycolmap import logging

from pycolmap_ba import core, utils

import enrich_data

# --- paths ------------------------------------------------------------------
NADIR = enrich_data.DATA_ROOT / "nadir-2"
image_path = NADIR / "images"
database_path = NADIR / "colmap/database.db"
sfm_path = NADIR / "colmap/sparse"  # free network, as the mapper leaves it
sfm_gcp_path = NADIR / "colmap/sparse_gcp"  # after the controlled bundle

# --- what to run ------------------------------------------------------------
# Feature extraction and brute-force matching take a while on CPU, so by default
# an existing database is reused.
FORCE_OVERWRITE_DB = True
# Give the *mapper* the camera centres as pose priors. False leaves the free
# network in an arbitrary frame, which the Sim3 below then fixes.
USE_MAPPER_POSE_PRIORS = False
# Constrain the camera centres in the GCP *bundle*, like Metashape's _GT project
# (all 39 references enabled). The camera-centre accuracy row then becomes a fit
# to data the bundle was given - the honest configuration is False.
USE_CAMERA_PRIORS = False
# Let the controlled bundle free the focal length. Safe here, and only here: the
# control is what breaks the flat-block degeneracy.
REFINE_INTRINSICS = True

logging.set_log_destination(logging.INFO, Path.cwd() / "INFO.log")

## 1. Data

`load()` reads the CSVs as they are; `add_noise()` returns another object of the
same type with a simulated survey applied. `truth` is kept for scoring, `data`
is what the pipeline is allowed to see.

In [2]:
truth = enrich_data.load(subblock="nadir-2", split_control_check=True)
data = truth  # or: enrich_data.add_noise(truth) to simulate a real survey
print(enrich_data.summary(data))

# --- names used further down ------------------------------------------------
surveyed_xyz = data.marker_xyz
marker_data = data.marker_data  # columns: image_name, marker_name, u, v
camera_centers = data.camera_xyz
marker_names = data.marker_names
control_names = data.control_names
check_names = data.check_names

# --- assumed accuracies (see docs: only the ratios matter) ------------------
tie_points_sigma_px = enrich_data.tie_points_sigma_px
gcp_sigma_px = enrich_data.gcp_sigma_px
gcp_sigma_m = enrich_data.gcp_sigma_m
camera_position_sigma_m = enrich_data.camera_position_sigma_m

print(
    f"survey noise: xy {gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {gcp_sigma_m[2] * 100:.0f} cm; marks {gcp_sigma_px} px"
    f"camera position noise: {camera_position_sigma_m * 100:.0f} cm, "
)
print(f"GCPs: {control_names}")
print(f"CP   {check_names}")

156 marker image marks, 26 markers, 39 images
control: 18 points | check: 8 points (gcp.001, gcp.004, gcp.010, gcp.013, gcp.016, gcp.019, gcp.022, gcp.025)
survey noise: xy 2 cm, z 3 cm; marks 1.0 pxcamera position noise: 100 cm, 
GCPs: ['gcp.002', 'gcp.003', 'gcp.005', 'gcp.006', 'gcp.007', 'gcp.008', 'gcp.009', 'gcp.011', 'gcp.012', 'gcp.014', 'gcp.015', 'gcp.017', 'gcp.018', 'gcp.020', 'gcp.021', 'gcp.023', 'gcp.024', 'gcp.026']
CP   ['gcp.001', 'gcp.004', 'gcp.010', 'gcp.013', 'gcp.016', 'gcp.019', 'gcp.022', 'gcp.025']


In [3]:
# Add some noise to the camera centres, to simulate a real survey.
rng = np.random.default_rng(42)
noise = rng.normal(0, camera_position_sigma_m, size=(len(camera_centers), 3))
camera_centers = {
    name: xyz + noise[i] for i, (name, xyz) in enumerate(camera_centers.items())
}

In [4]:
# Everything the reports need to score the block. Partial truth is fine: each
# field just enables one more row.
gt = utils.GroundTruth(
    camera_xyz=truth.camera_xyz,
    marker_xyz=truth.marker_xyz,
    control_names=truth.control_names,
    check_names=truth.check_names,
    cam_from_world=enrich_data.cam_from_world_all(truth),
    focal_px=enrich_data.FOCAL_PX,
)

## 2. Free network

Features, matches, known intrinsics, incremental mapping. Nothing here knows
about the GCPs yet.

In [22]:
# print(pycolmap.ImageReaderOptions().summary())
# print(pycolmap.FeatureExtractionOptions().summary())

In [6]:
database_path.parent.mkdir(parents=True, exist_ok=True)

reader_options = pycolmap.ImageReaderOptions(
    camera_model="SIMPLE_PINHOLE",
)
extraction_options = pycolmap.FeatureExtractionOptions(
    type=pycolmap.FeatureExtractorType.SIFT,
    max_image_size=-1,
    sift=pycolmap.SiftExtractionOptions(
        max_num_features=8192,
    ),
    use_gpu=False,
)
pycolmap.extract_features(
    database_path,
    image_path,
    camera_mode=pycolmap.CameraMode.SINGLE,
    reader_options=reader_options,
    extraction_options=extraction_options,
)


W20260816 21:07:56.670748 1681508 feature_extraction.cc:445] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20260816 21:07:56.671029 1681599 feature_extraction.cc:494] === Feature extraction ===
I20260816 21:07:56.671567 1681616 sift.cc:765] Creating SIFT CPU feature extractor
I20260816 21:07:56.671590 1681617 sift.cc:765] Creating SIFT CPU feature extractor
I20260816 21:07:56.671612 1681618 sift.cc:765] Creating SIFT CPU feature extractor
I20260816 21:07:56.671627 1681619 sift.cc:765] Creating SIFT CPU feature extractor
I20260816 21:07:56.671694 1681620 sift.cc:765] Creating SIFT CPU feature extractor
I20260816 21:07:56.671731 1681621 sift.cc:765] Creati

In [23]:
# print(pycolmap.FeatureMatchingOptions().summary())

In [ ]:
# Add Camera intrinsics to the database
with pycolmap.Database.open(Path(database_path)) as db:
    for entry in db.read_all_cameras():
        entry.model = pycolmap.CameraModelId.SIMPLE_PINHOLE
        entry.width = enrich_data.IMAGE_WIDTH
        entry.height = enrich_data.IMAGE_HEIGHT
        entry.params = [enrich_data.FOCAL_PX, *enrich_data.PRINCIPAL_POINT]
        entry.has_prior_focal_length = True
        db.update_camera(entry)
        print(f"  camera {entry.camera_id}: {entry.model.name} {entry.params}")


In [9]:
# Add camera pose priors to the database

sigma = np.broadcast_to(np.asarray(camera_position_sigma_m, dtype=float), (3,))
n_written = 0
with pycolmap.Database.open(Path(database_path)) as db:
    # Priors persist in the database across sessions, and writing a second
    # one for the same image violates a UNIQUE constraint - so reuse the
    # existing id and update in place instead.
    existing = {p.corr_data_id: p.pose_prior_id for p in db.read_all_pose_priors()}
    for image in db.read_all_images():
        if image.name not in camera_centers:
            continue
        prior = pycolmap.PosePrior()
        prior.corr_data_id = image.data_id
        prior.position = np.asarray(camera_centers[image.name], dtype=float)
        prior.position_covariance = np.diag(sigma**2)
        prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
        if image.data_id in existing:
            prior.pose_prior_id = existing[image.data_id]
            db.update_pose_prior(prior)
        else:
            db.write_pose_prior(prior)
        n_written += 1
    n_priors = db.num_pose_priors()

print(f"  wrote {n_written} pose priors, sigma {sigma} m")


  wrote 39 pose priors, sigma [1. 1. 1.] m


In [10]:
print(pycolmap.FeatureMatchingOptions().summary())
print(pycolmap.SpatialPairingOptions().summary())
print(pycolmap.TwoViewGeometryOptions().summary())

FeatureMatchingOptions:
    type = FeatureMatcherType.SIFT_BRUTEFORCE
    num_threads = -1
    use_gpu = True
    gpu_index = -1
    max_num_matches = 32768
    guided_matching = False
    skip_geometric_verification = False
    rig_verification = False
    skip_image_pairs_in_same_frame = False
    sift: SiftMatchingOptions:
        max_ratio = 0.8
        max_distance = 0.7
        cross_check = True
        cpu_brute_force_matcher = False
        lightglue: LightGlueONNXMatchingOptions:
            min_score = 0.1
            model_path = h ... 162 elements ... e
    aliked: AlikedMatchingOptions:
        brute_force: BruteForceONNXMatchingOptions:
            min_cossim = 0.85
            max_ratio = 1.0
            cross_check = True
            model_path = h ... 170 elements ... 7
        lightglue: LightGlueONNXMatchingOptions:
            min_score = 0.1
            model_path = h ... 166 elements ... d
SpatialPairingOptions:
    ignore_z = True
    max_num_neighbors = 50
    

In [11]:
matching_options = pycolmap.FeatureMatchingOptions(
    type=pycolmap.FeatureMatcherType.SIFT_BRUTEFORCE,
    use_gpu=False,
)
pairing_options = pycolmap.SpatialPairingOptions(
    ignore_z=True,
    max_num_neighbors=50,
    min_num_neighbors=0,
    max_distance=100.0,
    num_threads=-1,
)
two_view_geometry_options = pycolmap.TwoViewGeometryOptions(
    min_num_inliers=15,
    compute_relative_pose=True,
    ransac=pycolmap.RANSACOptions(
        max_error=2.0,
        confidence=0.9999,
    ),
)

# pycolmap.match_exhaustive(database_path, matching_options=matching_options)
pycolmap.match_spatial(
    database_path,
    matching_options=matching_options,
    pairing_options=pairing_options,
    verification_options=two_view_geometry_options,
)


I20260816 21:07:56.713228 1681633 feature_matching.cc:196] === Feature matching & geometric verification ===
I20260816 21:07:56.713465 1681637 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713478 1681638 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713455 1681634 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713449 1681636 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713536 1681641 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713447 1681639 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713459 1681635 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713570 1681642 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713608 1681643 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713626 1681640 sift.cc:1686] Creating SIFT CPU feature matcher
I20260816 21:07:56.713627 1681644 sift.cc:1686] Creating SIFT CPU featu

In [12]:
# print(pycolmap.IncrementalPipelineOptions().summary())

In [13]:
sfm_path.mkdir(parents=True, exist_ok=True)

pipeline_options = pycolmap.IncrementalPipelineOptions()
pipeline_options.use_prior_position = False
pipeline_options.use_robust_loss_on_prior_position = False
pipeline_options.prior_position_loss_scale = float(np.sqrt(7.815))
pipeline_options.ba_refine_focal_length = True
pipeline_options.ba_refine_principal_point = False
pipeline_options.ba_refine_extra_params = False
pipeline_options.mapper.filter_max_reproj_error = 1
pipeline_options.mapper.filter_min_tri_angle = 6
pipeline_options.mapper.ba_local_min_tri_angle = 6.0

with pycolmap.Database.open(database_path) as db:
    num_images = db.num_images()

with enlighten.Manager() as manager:
    with manager.counter(total=num_images, desc="Images registered:") as pbar:
        pbar.update(0, force=True)
        reconstructions = pycolmap.incremental_mapping(
            database_path,
            image_path,
            sfm_path,
            options=pipeline_options,
            initial_image_pair_callback=lambda: pbar.update(2),
            next_image_callback=lambda: pbar.update(1),
        )

if not reconstructions:
    raise RuntimeError(
        "incremental mapping produced no reconstruction - check the matches "
        "and, if pose priors are enabled, that there are >= 3"
    )
if len(reconstructions) > 1:
    sizes = [r.num_reg_images() for r in reconstructions.values()]
    print(f"  {len(reconstructions)} sub-models, sizes {sizes} -> keeping best")

rec = reconstructions[
    max(reconstructions, key=lambda k: reconstructions[k].num_reg_images())
]
print(rec.summary())
print("  camera:", next(iter(rec.cameras.values())).params)


Images registered:   0%|                               |  0/39 [00:00<?, 0.00/s]

I20260816 21:07:56.753968 1681508 incremental_pipeline.cc:348] Loading database
I20260816 21:07:56.754048 1681508 database_cache.cc:73] Loading rigs...
I20260816 21:07:56.754089 1681508 database_cache.cc:83]  1 in 0.000s
I20260816 21:07:56.754101 1681508 database_cache.cc:91] Loading cameras...
I20260816 21:07:56.754115 1681508 database_cache.cc:109]  1 in 0.000s
I20260816 21:07:56.754122 1681508 database_cache.cc:117] Loading frames...
I20260816 21:07:56.754179 1681508 database_cache.cc:127]  39 in 0.000s
I20260816 21:07:56.754186 1681508 database_cache.cc:135] Loading matches...
I20260816 21:07:56.759845 1681508 database_cache.cc:140]  254 in 0.006s
I20260816 21:07:56.759886 1681508 database_cache.cc:148] Loading images...
I20260816 21:07:56.773857 1681508 database_cache.cc:245]  39 in 0.014s (connected 39, loaded 39)
I20260816 21:07:56.773907 1681508 database_cache.cc:259] Loading pose priors...
I20260816 21:07:56.773940 1681508 database_cache.cc:267]  39 in 0.000s
I20260816 21:07:5

Images registered:   5%|█▍                         |  2/39 [00:00<00:09, 4.12/s]

I20260816 21:07:57.236838 1681508 incremental_pipeline.cc:607] Registering image #14 (num_reg_frames=2)
I20260816 21:07:57.236924 1681508 incremental_pipeline.cc:610] => Image sees 1594 / 7662 points
I20260816 21:07:57.583277 1681508 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment


Images registered:   8%|██▏                        |  3/39 [00:01<00:14, 2.49/s]

...

I20260816 21:08:59.811542 1681508 incremental_pipeline.cc:607] Registering image #39 (num_reg_frames=38)
I20260816 21:08:59.811578 1681508 incremental_pipeline.cc:610] => Image sees 3365 / 5350 points
I20260816 21:09:00.273814 1681508 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment


Images registered: 100%|███████████████████████████| 39/39 [01:10<00:00, 0.56/s]

I20260816 21:09:06.842976 1681508 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment


Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 39
	num_reg_frames = 39
	num_images = 39
	num_points3D = 37755
	num_observations = 176934
	mean_track_length = 4.68637
	mean_observations_per_image = 4536.77
Images registered: 100%|███████████████████████████| 39/39 [01:10<00:00, 0.56/s]
  camera: [4031.58583968 3008.         2008.        ]


I20260816 21:09:13.346943 1681508 incremental_pipeline.cc:793] Keeping successful reconstruction
I20260816 21:09:13.419248 1681508 timer.cc:90] Elapsed time: 1.276 [minutes]


In [14]:
# print(pycolmap.GlobalPipelineOptions().summary())

In [15]:
# sfm_path.mkdir(parents=True, exist_ok=True)

# pipeline_options = pycolmap.GlobalPipelineOptions()
# pipeline_options.mapper.global_positioning.use_gpu = False
# pipeline_options.mapper.bundle_adjustment.refine_focal_length = True
# pipeline_options.mapper.bundle_adjustment.refine_extra_params = False
# pipeline_options.mapper.bundle_adjustment.min_track_length = 3
# pipeline_options.mapper.bundle_adjustment.ceres.use_gpu = False
# pipeline_options.mapper.retriangulation.ignore_two_view_tracks = True
# reconstructions = pycolmap.global_mapping(
#     database_path,
#     image_path,
#     sfm_path,
#     options=pipeline_options,
# )

# if not reconstructions:
#     raise RuntimeError(
#         "incremental mapping produced no reconstruction - check the matches "
#         "and, if pose priors are enabled, that there are >= 3"
#     )
# if len(reconstructions) > 1:
#     sizes = [r.num_reg_images() for r in reconstructions.values()]
#     print(f"  {len(reconstructions)} sub-models, sizes {sizes} -> keeping best")

# rec = reconstructions[
#     max(reconstructions, key=lambda k: reconstructions[k].num_reg_images())
# ]
# print(rec.summary())
# print("  camera:", next(iter(rec.cameras.values())).params)

In [16]:
# Filter the observations to remove outliers, and optionally triangulate any missing

# Negative-depth points make the problem degenerate; COLMAP's own mapper
# drops them before every global bundle.
obs_manager = pycolmap.ObservationManager(rec)
n_filtered = 0
n_filtered += obs_manager.filter_observations_with_negative_depth()

# Filter large reprojection errors, small triangulation angles, short tracks.
n_filtered += obs_manager.filter_points3D_with_short_tracks(min_track_length=3)
n_filtered += obs_manager.filter_all_points3D(
    max_reproj_error=1,
    min_tri_angle=6,
)

print(f"  filtered {n_filtered} observations -> {rec.num_points3D():,} points remain")

  filtered 5776 observations -> 34,867 points remain


In [17]:
utils.banner("Free network vs the known camera centres")
rmse = utils.camera_center_rmse(rec, camera_centers)
print(f"camera centre RMSE: {rmse:.4f} m")

# A bundle adjustment is a local optimiser: if the network sits in an arbitrary
# frame, the GCP residuals start hopeless and the solve diverges.
# A least-squares similarity over all 39 centres brings it close enough.
if rmse > 5.0:
    utils.align_to_positions(rec, camera_centers)
    print(f"after Sim3: {utils.camera_center_rmse(rec, camera_centers):.4f} m")

# Write the free network to disk
utils.export_reconstruction(rec, sfm_path)


Free network vs the known camera centres
camera centre RMSE: 325.3300 m
  Sim3 over 39 centres, scale = 27.814803
after Sim3: 1.5277 m
  wrote bin + txt + ply to /home/francesco/dev/pycolmap_ba/data/ENRICH-Aerial/nadir-2/colmap/sparse


PosixPath('/home/francesco/dev/pycolmap_ba/data/ENRICH-Aerial/nadir-2/colmap/sparse')

In [18]:
# Scored before the control enters, so the effect of the GCPs is measurable.
# No adjusted marker coordinates exist yet, hence the empty dict: only the poses
# and the intrinsics can be scored at this point.
utils.block_report(rec)
utils.accuracy_report(rec, {}, gt, "free network")


BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [4031.586 3008.    2008.   ]
  points3D          34,867      observations 171,158
  track length      mean 4.91   median 4
                    2: 0 (0.0%)   3-5: 26,089 (74.8%)   6-9: 6,056 (17.4%)   10+: 2,722 (7.8%)
  observations/img  min 2,733   mean 4,389   max 5,751

  tie-point reprojection error (px)
    mean 0.299   median 0.277   RMS 0.345   p90 0.545   p99 0.703   max 0.94
    > 2 px: 0 points (0.0%)

  triangulation angle (deg, best pairwise ray angle per point)
    median 35.44   p10 19.84   < 1.5 deg: 0 points (0.0%)

  redundancy        342,316 equations - 104,838 unknowns  ->  r = 237,478  (no solver/result given - assumed all intrinsics free)
  sigma0_hat        0.315 px  (tie_points_sigma_px=0.50 px, ratio 0.630 - 1.0 = assumed sigmas about right)

ACCURACY vs GROUND TRUTH - free network
                                bias E   bias N   bias U    RMSE 3D    

## 3. GCP-controlled bundle adjustment

Each marker becomes a parameter block of its own, tied to the poses by its image
marks and - for the control points only - to the survey by a Gaussian prior. The
check points get rays but no prior, so their residual is the one unbiased
accuracy figure in the whole notebook.

All sigmas are given in native units (px, m); the backend expresses them in
tie-point sigma units, because COLMAP adds its tie-point residuals unweighted.

In [24]:
solverOptions = core.SolverOptions(
    refine_focal_length=REFINE_INTRINSICS,
    # Points with fewer rays would be IGNORED by the solver, not deleted:
    # they would keep pre-bundle coordinates and still reach the export.
    # 0 = disabled; the post-solve filter below is the one that deletes.
    min_track_length=3,
    # COLMAP's global-BA default. SOFT_L1 @ 1.0 would instead down-weight the
    # reprojection tail (p95 1.77, p99 2.27, max 3.98 px) without discarding
    # redundancy - the soft alternative to hard filtering.
    loss_function_type=pycolmap.LossFunctionType.SOFT_L1,
    # COLMAP ships function_tolerance = 0, which *disables* the relative
    # stopping test and makes every solve run to the iteration cap and report
    # NO_CONVERGENCE although the cost stopped improving. 1e-8 fixes it.
    function_tolerance=1e-8,
)
markersConstraints = core.MarkerConstraints(
    observations=marker_data,
    xyz=surveyed_xyz,
    control_names=control_names,  # the check points stay unconstrained
    sigma_px=gcp_sigma_px,
    sigma_m=gcp_sigma_m,
    robust_loss_scale=2.8,  # Cauchy on the marks, to absorb mis-identification
)

if USE_CAMERA_PRIORS:
    camera_posesConstraints = core.CameraPoseConstraints(
        positions=camera_centers, sigma_m=camera_position_sigma_m
    )
else:
    camera_posesConstraints = None

filterOptions = utils.FilterOptions(
    max_reproj_error=2.0,
    min_tri_angle_deg=1.5,
    min_track_length=3,
)

result = core.run_bundle_adjustment(
    rec,
    solver=solverOptions,
    markers=markersConstraints,
    camera_poses=camera_posesConstraints,
    # Post-solve filtering, same thresholds as the mapper: no-ops here, kept
    # explicit so every threshold in the notebook is visible in one place.
    filters=filterOptions,
    tie_points_sigma_px=tie_points_sigma_px,
    covariance=True,
    verbose=True,
)

tie points      : sigma 0.5 px  (reference unit)
marker image obs: sigma 1.0 px -> 2.00 in tie units
marker world    : sigma [0.02 0.02 0.03] m -> [0.04 0.04 0.06] in tie units
gauge           : None
  problem: 171,332 blocks = 171,158 tie points + 174 added (156 marker image, 18 marker world, 0 camera pose, 0 reference cloud)
  solve 1: BundleAdjustmentTerminationType.CONVERGENCE
  filtered 0 observations -> 34,867 points remain


In [25]:
marker_xyz = result.marker_xyz  # adjusted marker coordinates
utils.block_report(
    rec, result, marker_observations=marker_data, marker_xyz=marker_xyz
)
utils.accuracy_report(
    rec, marker_xyz, gt, "after GCP bundle", camera_priors_used=USE_CAMERA_PRIORS
)


BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.458 3008.    2008.   ]
  points3D          34,867      observations 171,158
  track length      mean 4.91   median 4
                    2: 0 (0.0%)   3-5: 26,089 (74.8%)   6-9: 6,056 (17.4%)   10+: 2,722 (7.8%)
  observations/img  min 2,733   mean 4,389   max 5,751

  tie-point reprojection error (px)
    mean 0.297   median 0.276   RMS 0.342   p90 0.538   p99 0.696   max 0.94
    > 2 px: 0 points (0.0%)

  triangulation angle (deg, best pairwise ray angle per point)
    median 26.28   p10 14.59   < 1.5 deg: 0 points (0.0%)

  redundancy        342,682 equations - 104,914 unknowns  ->  r = 237,768
  sigma0_hat        0.315 px  (tie_points_sigma_px=0.50 px, ratio 0.630 - 1.0 = assumed sigmas about right)

  solver            CONVERGED  (usable: True)
                    BundleAdjustmentTerminationType.CONVERGENCE
  block composition
    tie_points       171,158
    mark

In [ ]:
# Posterior point covariance, correctly scaled by sigma0_hat^2 - the raw
# `result.BACovariance.get_point_cov(id)` is an *unscaled* cofactor matrix
# (assumes tie_points_sigma_px was exactly right), not the real covariance.
# See docs/posterior_covariance.md. Saved to the ply cloud below.
pts3D_cov = result.point_covariances()

## 4. Export

Binary + text model for COLMAP, plus a point cloud carrying track length and
reprojection error per point so the sparse cloud can be filtered in a viewer.

In [27]:
utils.export_reconstruction(rec, sfm_gcp_path)
utils.save_point_cloud_as_ply(
    rec, sfm_gcp_path / "sparse_gcp_o3d.ply", point_covariances=pts3D_cov
)

  wrote bin + txt + ply to /home/francesco/dev/pycolmap_ba/data/ENRICH-Aerial/nadir-2/colmap/sparse_gcp
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


True